In [ ]:
pip install kaggle

In [ ]:
pip install ultralytics opencv-python numpy

In [ ]:
import os
import cv2
import shutil
import numpy as np
import kagglehub
from ultralytics import YOLO

# ===============================
# CONFIGURATION
# ===============================

TARGET_IMAGES = 800

RAW_DIR = "raw_images"
CLEAN_DIR = "clean_images"
REJECT_DIR = "rejected_images"

BLUR_THRESHOLD = 90

MIN_BOX_WIDTH_RATIO = 0.30
MIN_BOX_HEIGHT_RATIO = 0.25

ASPECT_RATIO_MIN = 0.6
ASPECT_RATIO_MAX = 2.6

BACKGROUND_EDGE_THRESHOLD = 0.18

CAR_CLASS_ID = 2

# ===============================
# DOWNLOAD DATASET
# ===============================

print("Downloading dataset...")

dataset_path = kagglehub.dataset_download(
    "kshitij192/cars-image-dataset"
)

print("Dataset downloaded at:", dataset_path)

# ===============================
# CREATE FOLDERS
# ===============================

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(CLEAN_DIR, exist_ok=True)
os.makedirs(REJECT_DIR, exist_ok=True)

# ===============================
# COPY IMAGES FROM DATASET
# ===============================

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        if file.lower().endswith((".jpg",".jpeg",".png")):

            src = os.path.join(root,file)
            dst = os.path.join(RAW_DIR,file)

            if not os.path.exists(dst):
                shutil.copy(src,dst)

print("Images copied to raw_images folder")

# ===============================
# LOAD YOLO MODEL
# ===============================

print("Loading YOLO model...")

model = YOLO("yolov8n.pt")

# ===============================
# HELPER FUNCTIONS
# ===============================

def is_blurry(img):

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    score = cv2.Laplacian(gray, cv2.CV_64F).var()

    return score < BLUR_THRESHOLD


def background_clutter_score(img):

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    edges = cv2.Canny(gray,100,200)

    density = np.sum(edges>0) / edges.size

    return density

# ===============================
# CLEAN DATASET GENERATION
# ===============================

print("Generating clean dataset...")

count = 0
img_id = 1

for file in os.listdir(RAW_DIR):

    if count >= TARGET_IMAGES:
        break

    if not file.lower().endswith((".jpg",".jpeg",".png")):
        continue

    path = os.path.join(RAW_DIR,file)

    img = cv2.imread(path)

    if img is None:
        continue

    h,w,_ = img.shape

    # 1 Blur Check
    if is_blurry(img):
        shutil.copy(path,REJECT_DIR)
        continue

    # 2 Background Check
    clutter = background_clutter_score(img)

    if clutter > BACKGROUND_EDGE_THRESHOLD:
        shutil.copy(path,REJECT_DIR)
        continue

    # 3 YOLO Detection
    results = model(img,conf=0.4,classes=[CAR_CLASS_ID])

    boxes = results[0].boxes

    if boxes is None or len(boxes)!=1:
        shutil.copy(path,REJECT_DIR)
        continue

    x1,y1,x2,y2 = map(int,boxes.xyxy[0])

    box_w = x2-x1
    box_h = y2-y1

    # 4 Full car visibility
    if box_w < MIN_BOX_WIDTH_RATIO*w or box_h < MIN_BOX_HEIGHT_RATIO*h:
        shutil.copy(path,REJECT_DIR)
        continue

    # 5 Front / Rear bias
    aspect = box_w/box_h

    if aspect < ASPECT_RATIO_MIN or aspect > ASPECT_RATIO_MAX:
        shutil.copy(path,REJECT_DIR)
        continue

    # 6 Save clean image
    new_name = f"clean_{img_id:04d}.jpg"

    shutil.copy(path, os.path.join(CLEAN_DIR,new_name))

    img_id += 1
    count += 1

print("===================================")
print("CLEAN DATASET GENERATED:",count)
print("===================================")

In [ ]:
import shutil
import os

CLEAN_FOLDER = "clean_images"
ZIP_NAME = "clean_car_dataset1"

if not os.path.exists(CLEAN_FOLDER):
    print("clean_images folder not found")
else:
    shutil.make_archive(ZIP_NAME, 'zip', CLEAN_FOLDER)
    print("Dataset zipped successfully!")

In [ ]:
# =================================== #
# FINAL VEHICLE CLASSIFICATION SYSTEM #
# =================================== #

!pip install -q opencv-python scikit-image xgboost imbalanced-learn seaborn gradio
!unzip -oq /content/FINAL.zip

import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gradio as gr

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler, LabelEncoder

from xgboost import XGBClassifier
from skimage.feature import graycomatrix, graycoprops
from imblearn.over_sampling import SMOTE

# =========================================================
# DATASET PATH
# =========================================================

dataset_path = "/content/fINAL_fINAL_DATASET"

# =========================================================
# AUGMENTATION
# =========================================================

def augment_image(img):
    aug = []
    aug.append(cv2.flip(img, 1))

    rows, cols, _ = img.shape
    M = cv2.getRotationMatrix2D((cols/2, rows/2), 15, 1)
    aug.append(cv2.warpAffine(img, M, (cols, rows)))

    bright = cv2.convertScaleAbs(img, alpha=1.2, beta=30)
    aug.append(bright)

    return aug

# =========================================================
# PREPROCESSING
# =========================================================

def preprocess_image(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5,5), 0)

    clahe = cv2.createCLAHE(2.0, (8,8))
    contrast = clahe.apply(blur)

    edges = cv2.Canny(contrast, 30, 100)

    return contrast, edges

# =========================================================
# SEGMENTATION
# =========================================================

def segment_vehicle(edges):
    _, thresh = cv2.threshold(edges, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel = np.ones((3,3), np.uint8)
    return cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=2)

# =========================================================
# FEATURES
# =========================================================

def shape_features(contour):
    x,y,w,h = cv2.boundingRect(contour)

    aspect_ratio = w/float(h)
    area = cv2.contourArea(contour)
    perimeter = cv2.arcLength(contour, True)

    rect_area = w*h
    extent = area/rect_area if rect_area!=0 else 0

    hull = cv2.convexHull(contour)
    hull_area = cv2.contourArea(hull)
    solidity = area/hull_area if hull_area!=0 else 0

    hu = cv2.HuMoments(cv2.moments(contour)).flatten()

    return [w,h,aspect_ratio,area,perimeter,extent,solidity] + list(hu)

def texture_features(image):
    glcm = graycomatrix(image,[1,2],[0,np.pi/4,np.pi/2],256,True,True)

    return [
        graycoprops(glcm,'contrast').mean(),
        graycoprops(glcm,'energy').mean(),
        graycoprops(glcm,'homogeneity').mean(),
        graycoprops(glcm,'correlation').mean()
    ]

def color_features(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    return [
        np.mean(hsv[:,:,0]),
        np.mean(hsv[:,:,1]),
        np.mean(hsv[:,:,2])
    ]

def edge_density(edges):
    return np.sum(edges>0)/edges.size

# =========================================================
# SUV FEATURES (UNCHANGED)
# =========================================================

def suv_hatch_features(gray, contour):

    x,y,w,h = cv2.boundingRect(contour)
    roi = gray[y:y+h, x:x+w]

    if roi.size == 0:
        return [0,0,0,0,0]

    roi = cv2.resize(roi, (120,120))

    top = roi[:40,:]
    middle = roi[40:80,:]
    bottom = roi[80:,:]

    vertical_ratio = np.mean(top) / (np.mean(bottom) + 1e-5)
    height_ratio = h / (w + 1e-5)

    edges_roi = cv2.Canny(roi, 50, 150)
    top_edges = np.sum(edges_roi[:40,:] > 0)
    mid_edges = np.sum(edges_roi[40:80,:] > 0)
    bottom_edges = np.sum(edges_roi[80:,:] > 0)

    edge_ratio = top_edges / (bottom_edges + 1e-5)

    if y+h+20 < gray.shape[0]:
        ground_strip = gray[y+h:y+h+20, x:x+w]
        ground_clearance = np.mean(ground_strip)
    else:
        ground_clearance = 0

    return [vertical_ratio, height_ratio, edge_ratio, mid_edges, ground_clearance]

# =========================================================
# 🔥 FEATURE EXTRACTION (ONLY SAFE CHANGE)
# =========================================================

def extract_features(img):

    gray, edges = preprocess_image(img)
    seg = segment_vehicle(edges)

    contours,_ = cv2.findContours(seg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # 🔥 slightly increased threshold (safe improvement)
    contours = [c for c in contours if cv2.contourArea(c) > 800]

    # 🔥 NO CAR DETECTION
    if len(contours) == 0 or edge_density(edges) < 0.005:
        return None

    c = max(contours, key=cv2.contourArea)

    shape = shape_features(c)
    texture = texture_features(gray)
    color = color_features(img)
    special = suv_hatch_features(gray, c)

    final = shape + texture + color + [edge_density(edges), len(contours)] + special

    return np.array(final).reshape(1,-1)

# =========================================================
# TRAINING (UNCHANGED)
# =========================================================

features, labels = [], []

for label in os.listdir(dataset_path):

    folder = os.path.join(dataset_path, label)
    if not os.path.isdir(folder):
        continue

    for file in os.listdir(folder):

        img = cv2.imread(os.path.join(folder,file))
        if img is None:
            continue

        all_imgs = [img] + augment_image(img)

        for im in all_imgs:

            feat = extract_features(im)
            if feat is None:
                continue

            features.append(feat.flatten())
            labels.append(label)

df = pd.DataFrame(features)
df["label"] = labels

print("Dataset Size:", df.shape)

X = df.drop("label",axis=1)
y = df["label"]

le = LabelEncoder()
y_enc = le.fit_transform(y)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train,X_test,y_train,y_test = train_test_split(
    X_scaled,y_enc,test_size=0.2,stratify=y_enc,random_state=42
)

smote = SMOTE()
X_train, y_train = smote.fit_resample(X_train, y_train)

model = XGBClassifier(
    n_estimators=400,
    max_depth=10,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    gamma=0.2,
    eval_metric='mlogloss'
)

model.fit(X_train,y_train)

# =========================================================
# EVALUATION (RESTORED)
# =========================================================

y_pred = model.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test,y_pred)*100)
print("\nReport:\n", classification_report(y_test,y_pred,target_names=le.classes_))

cm = confusion_matrix(y_test,y_pred)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_,
            yticklabels=le.classes_)

plt.title("Confusion Matrix")
plt.show()

# =========================================================
# PREDICTION
# =========================================================

def predict_vehicle(image):

    img = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

    feat = extract_features(img)

    if feat is None:
        return "❌ No Car Found"

    feat_scaled = scaler.transform(feat)

    pred = model.predict(feat_scaled)
    probs = model.predict_proba(feat_scaled)

    label = le.inverse_transform(pred)[0]
    conf = np.max(probs)

    # =========================================================
    # 🔥 ULTRA-SAFE CORRECTION (ONLY WHEN VERY NECESSARY)
    # =========================================================

    height_ratio = feat.flatten()[-4]
    ground_clearance = feat.flatten()[-1]

    # ONLY fix when:
    # 1. Model is confused
    # 2. SUV features are VERY strong

    if conf < 0.60:
        if height_ratio > 1.0 and ground_clearance > 100:
            label = "SUV"

    # Otherwise DO NOT TOUCH prediction

    return f"{label} ({round(conf*100,2)}%)"
# =========================================================
# UI
# =========================================================

gr.Interface(
    fn=predict_vehicle,
    inputs=gr.Image(type="numpy"),
    outputs="text",
    title="🚗 Vehicle Classifier (Stable Final)",
).launch()